# 🎵 Music Hit Predictor: Indonesian Market Pipeline (Manual Extraction Version)
### *Leveraging Librosa Audio Features & Machine Learning to Predict Hits*

--- 
## 📌 Project Overview
Since Spotify's audio-features API is restricted, we've transitioned to a **Manual Extraction Flow**. We use **Librosa** to extract features like Tempo, MFCC, and Spectral Centroid from raw audio, then train a classification model to predict "Hit" potential.

### 🧪 The Pipeline Strategy
1. **Ingestion:** Loading the manual features from JSON.
2. **Preprocessing:** Scaling and Balancing via SMOTE.
3. **The Big 5 Model Arena:** Benchmarking XGBoost, Logistic Regression, SVM, Random Forest, and KNN.
4. **Export:** Saving model artifacts (.pkl) for backend use.

---

## 1. Data Collection & Ingestion

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

sns.set_theme(style="whitegrid", palette="muted")

def load_master_data():
    path = "indo_music_sample.json"
    if not os.path.exists(path):
        print(f"❌ ERROR: {path} not found!")
        return pd.DataFrame()
        
    with open(path, 'r') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data)
    print(f"✅ Ingested: {path} | Total Records: {len(df):,}")
    return df

df_master = load_master_data()
if not df_master.empty:
    display(df_master.head())
    print(f"\nTarget Distribution:\n{df_master['target'].value_counts()}")

## 2. Professional Data Preprocessing
*Utilizing 17 Librosa-extracted features.*

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# 1. Select relevant Librosa features
FEATURES = [
    'tempo', 'spectral_centroid', 'zcr', 'energy',
    'mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 
    'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 
    'mfcc_11', 'mfcc_12', 'mfcc_13'
]

X = df_master[FEATURES]
y = df_master['target']

# 2. Split Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2026, stratify=y
)

# 3. Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Balance via SMOTE
smote = SMOTE(random_state=2026)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

if not os.path.exists('models'): os.makedirs('models')
joblib.dump(scaler, 'models/audio_scaler.pkl')
print("✅ Scaler saved to models/audio_scaler.pkl")

## 3. The Big 5 Model Arena
Benchmarking the algorithms to find the champion.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=2026),
    "XGBoost": XGBClassifier(eval_metric='logloss'),
    "SVM": SVC(probability=True, kernel='rbf', random_state=2026),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results_metrics = []

for name, model in models.items():
    print(f"🔥 Training {name}...")
    model.fit(X_train_balanced, y_train_balanced)
    preds = model.predict(X_test_scaled)
    
    # Export each model
    save_path = f"models/{name.lower().replace(' ', '_')}_model.pkl"
    joblib.dump(model, save_path)
    
    results_metrics.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds)
    })

df_results = pd.DataFrame(results_metrics).set_index("Model").sort_values("F1-Score", ascending=False)
display(df_results.style.background_gradient(cmap='YlGnBu').format("{:.2%}"))